# R0 — TPSMM baseline on LSA64 (the M1 gate)

**The first thing in this project that spends GPU quota.** Everything before
it ran on free CPU sessions.

Target (paper Table 1, LSA64): L1 0.01342 | SSIM 0.9208 | LPIPS 0.02261 |
FVD 182.785 | TCD 0.130.

M1 is a hard gate. If TPSMM does not reproduce its own published row, we stop
and investigate rather than proceeding to PGMM — a PGMM number sitting on an
unvalidated baseline measures nothing.

## This run's job is to measure, not to finish

It trains a bounded number of steps and reports **seconds/step** (D12) — the
number the spec's 8–12 week estimate rests on and which has never been
measured. D18 suggests that estimate may be badly pessimistic: an epoch is
2800×5 = 14,000 samples, so the full 100 epochs is only ~50,000 steps.

Do not launch the full run until this reports and the arithmetic is redone.

## Data

Mounted from the preprocess kernel via `kernel_sources` — Kaggle bundles that
kernel's output into a single `_output_.zip`, which we extract here (D20).
Nothing transfers through the workstation, and this kernel authenticates with
nothing (D16).

In [ ]:
# 1. hardware. Two T4s must both be visible: quota bills session wall-clock,
#    not GPU-hours, so a session using one GPU wastes half the quota it costs.
import subprocess, sys, time

import torch

print("torch  ", torch.__version__, "| cuda", torch.version.cuda)
n = torch.cuda.device_count()
print("gpus   ", n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}")
assert n == 2, f"expected 2 T4s, got {n} -- fix the notebook's accelerator setting"

# Turing: fp16 tensor cores yes, bf16 no, flash-attn no (needs Ampere+).
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# 2. code
r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
sys.path.insert(0, "/kaggle/working/repo")
print("repo cloned")

In [ ]:
# 3. data: extract the preprocess kernel's output in place (D20).
import zipfile
from pathlib import Path

zips = sorted(Path("/kaggle/input").rglob("_output_.zip"))
assert zips, "no _output_.zip mounted -- is kernel_sources set to pgmm-preprocess?"
print("mounted archive:", zips[0], f"({zips[0].stat().st_size/2**30:.2f} GiB)")

DATA = Path("/kaggle/working/data")
t0 = time.time()
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATA)
print(f"extracted in {(time.time()-t0)/60:.1f} min")

ROOT = DATA / "lsa64_prepared"
n_train = len(list((ROOT / "train").iterdir()))
n_test = len(list((ROOT / "test").iterdir()))
print("train clips:", n_train, "| test clips:", n_test)
assert (n_train, n_test) == (2800, 400), (
    f"paper says 2800/400, got {n_train}/{n_test}. Without train/ and test/ on "
    f"disk, FramesDataset silently does its own 80/20 split (D17)."
)

In [ ]:
# 4. config: point it at the extracted data, and cap the run so this session
#    measures rather than trains. Everything else is the committed config.
import yaml

CFG_SRC = Path("/kaggle/working/repo/pgmm/config/lsa64-tpsmm.yaml")
cfg = yaml.safe_load(CFG_SRC.read_text())
cfg["dataset_params"]["root_dir"] = str(ROOT)

MEASURE_EPOCHS = 1          # one epoch = 500 steps at batch 28 -- enough to time
cfg["train_params"]["num_epochs"] = MEASURE_EPOCHS
cfg["train_params"]["checkpoint_freq"] = 1

CFG = Path("/kaggle/working/lsa64-tpsmm-measure.yaml")
CFG.write_text(yaml.safe_dump(cfg, sort_keys=False))

t = cfg["train_params"]
steps_per_epoch = 2800 * t["num_repeats"] / t["batch_size"]
print(f"batch_size     : {t['batch_size']}")
print(f"num_repeats    : {t['num_repeats']}  (D18: paper's 'five pairs per video')")
print(f"steps/epoch    : {steps_per_epoch:.0f}")
print(f"full run       : {steps_per_epoch*100:,.0f} steps over 100 epochs")
print(f"this run       : {MEASURE_EPOCHS} epoch(s) to measure s/step")

In [ ]:
# 6. D12: the measurement this session exists for. Redo the timeline arithmetic
#    from a real number instead of the spec's unmeasured guess.
#
# Refuse to compute from a failed run. The first attempt of this notebook died
# in 0.2 min on an ImportError and this cell cheerfully reported "0.026 s/step
# -> 6-run matrix: 2 hours". A number derived from a crash is worse than no
# number: it looks like evidence.
assert r.returncode == 0, (
    f"training exited {r.returncode} -- there is no step time to measure. "
    f"Fix the failure before reading anything into the wall clock."
)
assert elapsed > 60, (
    f"training returned 0 after only {elapsed:.1f}s; that is too fast to be a "
    f"real epoch of {steps_per_epoch:.0f} steps -- suspect a silent no-op."
)

s_per_step = elapsed / steps_per_epoch / MEASURE_EPOCHS
hours_per_run = s_per_step * steps_per_epoch * 100 / 3600

print(f"measured        : {s_per_step:.3f} s/step  (includes startup, so pessimistic)")
print(f"full 100 epochs : {hours_per_run:.1f} GPU-hours per run")
print(f"sessions/run    : {hours_per_run/9:.1f}  (9h session cap)")
print(f"6-run matrix    : {hours_per_run*6:.0f} hours")
print(f"weeks of quota  : {hours_per_run*6/30:.1f}  (30h/week)")
print()
print("Spec assumed 10-20h/run -> 8-12 weeks. Record the real figure as D12.")
print("If the 6-run matrix exceeds ~8 weeks of quota, stop and raise it: the")
print("options (fewer ablations, smaller batch, AMP, paid compute) are the")
print("user's call, not the implementer's.")

for p in sorted(Path("/kaggle/working/log").rglob("*.pth"))[:5]:
    print("checkpoint:", p.name, f"{p.stat().st_size/2**20:.0f} MiB")

In [ ]:
# 6. D12: the measurement this session exists for. Redo the timeline arithmetic
#    from a real number instead of the spec's unmeasured guess.
s_per_step = elapsed / steps_per_epoch / MEASURE_EPOCHS
hours_per_run = s_per_step * steps_per_epoch * 100 / 3600

print(f"measured        : {s_per_step:.3f} s/step  (includes startup, so pessimistic)")
print(f"full 100 epochs : {hours_per_run:.1f} GPU-hours per run")
print(f"sessions/run    : {hours_per_run/9:.1f}  (9h session cap)")
print(f"6-run matrix    : {hours_per_run*6:.0f} hours")
print(f"weeks of quota  : {hours_per_run*6/30:.1f}  (30h/week)")
print()
print("Spec assumed 10-20h/run -> 8-12 weeks. Record the real figure as D12.")
print("If the 6-run matrix exceeds ~8 weeks of quota, stop and raise it: the")
print("options (fewer ablations, smaller batch, AMP, paid compute) are the")
print("user's call, not the implementer's.")

for p in sorted(Path("/kaggle/working/log").rglob("*.pth"))[:5]:
    print("checkpoint:", p.name, f"{p.stat().st_size/2**20:.0f} MiB")

In [ ]:
# 7. keep the output small: the extracted data must not be republished, and
#    the repo clone would go with it.
import shutil

shutil.rmtree(DATA, ignore_errors=True)
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
print("output root:", sorted(p.name for p in Path("/kaggle/working").iterdir()))